Selected features of dataset cicids2017_cleaned.csv 
1.  Bwd Packet Length Std 
2.  Subflow Fwd Bytes 
3.  Flow Duration
4.  Total Length of Fwd Packets
5.  Init_Win_bytes_forward
6.  Flow IAT Std          
7.  Active Mean           
8.  Bwd Packets/s         
9.  Fwd Packet Length Mean
10. Bwd Packet Length Min 
11. Attack Type (label)          

The dataset is shuffled and divided into three datasets:
1. train_data.csv for training
2. val_data.csv for validation
3. test_data.csv for testing

Import pandas library

In [41]:
import pandas as pd

Read datasets.

In [42]:
label_column = "Attack Type"

train = pd.read_csv("train_data.csv")
train_x = train.drop(label_column, axis=1)
train_y = train[label_column]

validation = pd.read_csv("val_data.csv")
validation_x = validation.drop(label_column, axis=1)
validation_y = validation[label_column]

test = pd.read_csv("test_data.csv")
test_x = test.drop(label_column, axis=1)
test_y = test[label_column]

Scale features, then train and tune kNN using the validation set. Fit the kNN model. Use validation data to fine tune the model. Compare scalers and kNN parameter sets, then keep the best validation result.

In [43]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from time import perf_counter

scalers = {
    "standard": StandardScaler(),
    "minmax": MinMaxScaler(),
}

smote_options = [False, True]

param_grid = [
    {"n_neighbors": 3, "weights": "uniform", "metric": "euclidean"},
    {"n_neighbors": 5, "weights": "uniform", "metric": "euclidean"},
    {"n_neighbors": 5, "weights": "distance", "metric": "euclidean"},
    {"n_neighbors": 7, "weights": "distance", "metric": "euclidean"},
    {"n_neighbors": 9, "weights": "distance", "metric": "manhattan"},
]

results = []
best_result = None

for scaler_name, scaler in scalers.items():
    scaled_train_x = scaler.fit_transform(train_x)
    scaled_validation_x = scaler.transform(validation_x)

    for use_smote in smote_options:
        smote_time = 0.0
        x_train_used = scaled_train_x
        y_train_used = train_y

        if use_smote:
            smote_start = perf_counter()
            smote = SMOTE(random_state=42, k_neighbors=3)
            x_train_used, y_train_used = smote.fit_resample(scaled_train_x, train_y)
            smote_time = perf_counter() - smote_start

        for params in param_grid:
            fit_start = perf_counter()
            knn = KNeighborsClassifier(**params)
            knn.fit(x_train_used, y_train_used)
            fit_time = perf_counter() - fit_start

            predict_start = perf_counter()
            y_pred = knn.predict(scaled_validation_x)
            predict_time = perf_counter() - predict_start
            acc = accuracy_score(validation_y, y_pred)

            result = {
                "model_name": "kNN",
                "scaler_name": scaler_name,
                "scaler": scaler,
                "smote_used": use_smote,
                "params": params,
                "accuracy": acc,
                "smote_time_sec": smote_time,
                "fit_time_sec": fit_time,
                "predict_time_sec": predict_time,
                "total_time_sec": smote_time + fit_time + predict_time,
                "train_rows_used": len(y_train_used),
                "confusion_matrix": confusion_matrix(validation_y, y_pred),
                "report": classification_report(validation_y, y_pred),
                "model": knn,
            }
            results.append(result)

            if best_result is None or acc > best_result["accuracy"]:
                best_result = result

            print(f"\nScaler: {scaler_name}, SMOTE: {use_smote}, params: {params}")
            print("accuracy:", acc)
            print("smote time (sec):", smote_time)
            print("fit time (sec):", fit_time)
            print("predict time (sec):", predict_time)
            print(result["confusion_matrix"])
            print(result["report"])


Scaler: standard, SMOTE: False, params: {'n_neighbors': 3, 'weights': 'uniform', 'metric': 'euclidean'}
accuracy: 0.99295
smote time (sec): 0.0
fit time (sec): 0.13665596000009828
predict time (sec): 0.5030335700002979
[[    9     0     0     0     8     0     0]
 [    0    68     0     0     8     0     0]
 [    0     0  1013     7     3     0     0]
 [    0     0     7  1506    21     0     0]
 [    7     3    10    22 16542    31     1]
 [    0     0     0     1     5   709     0]
 [    0     0     0     0     7     0    12]]
                precision    recall  f1-score   support

          Bots       0.56      0.53      0.55        17
   Brute Force       0.96      0.89      0.93        76
          DDoS       0.98      0.99      0.99      1023
           DoS       0.98      0.98      0.98      1534
Normal Traffic       1.00      1.00      1.00     16616
 Port Scanning       0.96      0.99      0.97       715
   Web Attacks       0.92      0.63      0.75        19

      accuracy

Select best combination of parameters and scaler.

In [44]:
best_combination = max(range(len(results)), key=lambda i: results[i]["accuracy"])
best_result = results[best_combination]

best_scaler = best_result["scaler"]
best_params = best_result["params"]
best_smote_used = best_result["smote_used"]

scaled_train_x = best_scaler.fit_transform(train_x)
scaled_validation_x = best_scaler.transform(validation_x)

x_train_used = scaled_train_x
y_train_used = train_y
if best_smote_used:
    smote = SMOTE(random_state=42, k_neighbors=3)
    x_train_used, y_train_used = smote.fit_resample(scaled_train_x, train_y)

best_knn = KNeighborsClassifier(**best_params)
best_knn.fit(x_train_used, y_train_used)

print("Best configuration:")
print("Scaler:", best_result["scaler_name"])
print("SMOTE:", best_smote_used)
print("Params:", best_params)

Best configuration:
Scaler: standard
SMOTE: False
Params: {'n_neighbors': 9, 'weights': 'distance', 'metric': 'manhattan'}


In [45]:
import pandas as pd
from IPython.display import display

results_table = pd.DataFrame([
    {
        "model": item["model_name"],
        "scaler": item["scaler_name"],
        "smote_used": item["smote_used"],
        "n_neighbors": item["params"]["n_neighbors"],
        "weights": item["params"]["weights"],
        "metric": item["params"]["metric"],
        "accuracy": item["accuracy"],
        "smote_time_sec": item["smote_time_sec"],
        "fit_time_sec": item["fit_time_sec"],
        "predict_time_sec": item["predict_time_sec"],
        "total_time_sec": item["total_time_sec"],
        "train_rows_used": item["train_rows_used"],
    }
    for item in results
])

results_table = results_table.sort_values(["accuracy", "total_time_sec"], ascending=[False, True])

print("All results ordered by accuracy and total time:")
display(results_table)
print("Best configuration:")
print(best_result["scaler_name"], "SMOTE:", best_result["smote_used"], best_result["params"])

All results ordered by accuracy and total time:


,model,scaler,smote_used,n_neighbors,weights,metric,accuracy,smote_time_sec,fit_time_sec,predict_time_sec,total_time_sec,train_rows_used
4,kNN,standard,False,9,distance,manhattan,0.99380,0.000000,0.124139,0.827264,0.951403,60000
2,kNN,standard,False,5,distance,euclidean,0.99330,0.000000,0.117345,0.578338,0.695683,60000
0,kNN,standard,False,3,uniform,euclidean,0.99295,0.000000,0.136656,0.503034,0.639690,60000
3,kNN,standard,False,7,distance,euclidean,0.99255,0.000000,0.118326,0.620188,0.738514,60000
14,kNN,minmax,False,9,distance,manhattan,0.99250,0.000000,0.118961,0.490885,0.609846,60000
12,kNN,minmax,False,5,distance,euclidean,0.99245,0.000000,0.115898,0.360516,0.476414,60000
13,kNN,minmax,False,7,distance,euclidean,0.99210,0.000000,0.116406,0.388826,0.505232,60000
10,kNN,minmax,False,3,uniform,euclidean,0.99175,0.000000,0.115882,0.301932,0.417814,60000
1,kNN,standard,False,5,uniform,euclidean,0.99150,0.000000,0.118731,0.579625,0.698356,60000
11,kNN,minmax,False,5,uniform,euclidean,0.98980,0.000000,0.118233,0.345278,0.463511,60000


Best configuration:
standard SMOTE: False {'n_neighbors': 9, 'weights': 'distance', 'metric': 'manhattan'}


Evaluate the best kNN model on the test set.

In [46]:
scaled_test_x = best_scaler.transform(test_x)
y_pred = best_knn.predict(scaled_test_x)

print("Confusion Matrix:")
print(confusion_matrix(test_y, y_pred))
print("\nClassification Report:")
print(classification_report(test_y, y_pred))

Confusion Matrix:
[[    4     0     0     0    14     0     0]
 [    0    70     0     1     4     0     0]
 [    0     0  1016     5     2     0     0]
 [    0     1     1  1514    19     0     0]
 [    7     2     8    26 16533    40     0]
 [    0     0     0     2     2   711     0]
 [    0     0     0     0    12     0     6]]

Classification Report:
                precision    recall  f1-score   support

          Bots       0.36      0.22      0.28        18
   Brute Force       0.96      0.93      0.95        75
          DDoS       0.99      0.99      0.99      1023
           DoS       0.98      0.99      0.98      1535
Normal Traffic       1.00      1.00      1.00     16616
 Port Scanning       0.95      0.99      0.97       715
   Web Attacks       1.00      0.33      0.50        18

      accuracy                           0.99     20000
     macro avg       0.89      0.78      0.81     20000
  weighted avg       0.99      0.99      0.99     20000

